# Notebook 4 — Anomaly Detection in AV Trajectories

Identifies structurally unusual driving behaviour using both statistical
and ML-based detectors:

- Z-score detector (per-segment normalisation)
- CUSUM change-point detection
- Mahalanobis distance (multivariate statistical outlier)
- Isolation Forest
- One-Class SVM
- Feature importance analysis

In [1]:
import sys
sys.path.insert(0, '..')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from src.data.waymo_loader import load_dataset, extract_trajectories
from src.metrics.safety_metrics import nearest_agent_distances
from src.anomaly_detection.statistical_detector import combined_statistical_anomalies
from src.anomaly_detection.ml_detector import run_ml_detectors, anomaly_feature_importance
from src.visualization.plots import plot_anomaly_scores, plot_feature_importance

sns.set_theme(style='whitegrid')

In [2]:
dataset = load_dataset('../data', max_segments=5)
trajectories = extract_trajectories(dataset['lidar_box'])
trajectories = nearest_agent_distances(trajectories)
print(f'Loaded {len(trajectories):,} trajectory rows')

Loaded 62,312 trajectory rows


## Statistical Anomaly Detection

In [3]:
df_stat = combined_statistical_anomalies(trajectories, zscore_threshold=3.5)
print(f'Statistical anomalies: {df_stat["statistical_anomaly"].sum():,} / {len(df_stat):,}')
print(f'  Z-score: {df_stat["zscore_anomaly"].sum():,}')
print(f'  CUSUM:   {df_stat["cusum_anomaly"].sum():,}')
print(f'  Mahal:   {df_stat["mahal_anomaly"].sum():,}')

Statistical anomalies: 15,484 / 62,312
  Z-score: 2,635
  CUSUM:   13,396
  Mahal:   2,924


In [4]:
# Show the most anomalous rows by max z-score
z_cols = [c for c in df_stat.columns if c.startswith('z_')]
df_stat.nlargest(10, 'zscore_max_z')[['segment_id', 'object_id', 'timestamp_s',
                                       'speed', 'acceleration', 'jerk', 'zscore_max_z']]

,segment_id,object_id,timestamp_s,speed,acceleration,jerk,zscore_max_z
31720,10289507859301986274_4200_000_4220_000,aC2W_twppVlfs_zvdg7LDg,1.557847e+09,10.034620,0.000000e+00,-2.401064e+01,65.045998
8067,1024360143612057520_3580_000_3600_000,Af7ZIfZSdSyhCjj2hGWXug,1.553736e+09,4.297226,0.000000e+00,-1.422134e+01,58.742775
57769,10289507859301986274_4200_000_4220_000,5e7riSzEhapbd6-zOb0mMg,1.557847e+09,1.439133,0.000000e+00,-1.277671e+01,34.608099
10376,1024360143612057520_3580_000_3600_000,kbQ8U7BVfmJLNKQ83VNnJA,1.553736e+09,7.005979,0.000000e+00,-7.791902e+00,32.187119
20961,1024360143612057520_3580_000_3600_000,QXNmwwTbadPIfINsMgU5IA,1.553736e+09,0.013744,7.203391e-03,3.307926e-02,27.796717
22795,1024360143612057520_3580_000_3600_000,2g5bmO0Ima-X3Ru03wmIGw,1.553736e+09,0.006238,3.278517e-02,3.223382e-02,27.795949
24448,1024360143612057520_3580_000_3600_000,DtnWLMgr1sZWJoQ7t9XAow,1.553736e+09,1.223200,2.196929e-01,-9.301431e-04,27.794285
24702,1024360143612057520_3580_000_3600_000,ecChntPq8IoUX-kEYyY7xw,1.553736e+09,0.018361,0.000000e+00,-3.244721e-11,27.794012
24649,1024360143612057520_3580_000_3600_000,8dEx1v1VyHatiD1JPVz3-w,1.553736e+09,0.042653,6.967421e-02,-2.175570e-05,27.794012
24862,1024360143612057520_3580_000_3600_000,G6WMCo3hsmuDoP29dRoKRg,1.553736e+09,0.619737,4.364978e-12,-1.095346e-14,27.793199


## ML-based Anomaly Detection — Isolation Forest

In [5]:
df_ml = run_ml_detectors(trajectories, use_ocsvm=False)
print(f'Isolation Forest anomalies: {df_ml["if_anomaly"].sum():,} / {len(df_ml):,}')
print(f'Anomaly rate: {df_ml["if_anomaly"].mean()*100:.2f}%')

Isolation Forest anomalies: 1,247 / 62,312
Anomaly rate: 2.00%


In [6]:
fig = plot_anomaly_scores(df_ml, score_col='if_score', anomaly_col='if_anomaly')
plt.show()

/var/folders/jj/khf1rsjs76137181nr0hly600000gn/T/ipykernel_27824/4158353487.py:2: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## Feature Importance

In [7]:
importances = anomaly_feature_importance(trajectories)
print('Feature importances (Isolation Forest):')
print(importances)

fig = plot_feature_importance(importances)
plt.show()

Feature importances (Isolation Forest):
heading_rate          0.219431
jerk                  0.203432
nearest_agent_dist    0.195272
speed                 0.193049
acceleration          0.188816
dtype: float64


/var/folders/jj/khf1rsjs76137181nr0hly600000gn/T/ipykernel_27824/1933243760.py:6: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## Anomaly Profile: What makes an anomalous frame different?

In [8]:
features = ['speed', 'acceleration', 'jerk', 'heading_rate', 'nearest_agent_dist']
present = [f for f in features if f in df_ml.columns]

normal = df_ml[~df_ml['if_anomaly']]
anomalies = df_ml[df_ml['if_anomaly']]

comparison = pd.DataFrame({
    'Normal (mean)': normal[present].mean(),
    'Anomaly (mean)': anomalies[present].mean(),
}).T

comparison.plot(kind='bar', figsize=(12, 5), edgecolor='white')
plt.title('Feature Profile: Normal vs Anomalous Frames')
plt.ylabel('Mean value')
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

/var/folders/jj/khf1rsjs76137181nr0hly600000gn/T/ipykernel_27824/118110530.py:17: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
